# Bộ đặc trưng HHL — HOG + Histogram + Landmarks

| # | Đặc trưng | Phương pháp | Chiều vector |
|---|---|---|---|
| 1 | **HOG** | Histogram of Oriented Gradients — orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), L2-Hys | ~3780 |
| 2 | **Color Histogram** | HSV 3D histogram 8×8×8 bins, normalize MinMax → [0,1] | 512 |
| 3 | **Landmarks** | MediaPipe Face Mesh: 468 điểm (x, y) chuẩn hóa | 936 |

> Mỗi mục minh họa **TRƯỚC** (ảnh đầu vào) và **SAU** (đặc trưng trích xuất).

In [ ]:
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage.feature import hog
import mediapipe as mp

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 11, 'font.size': 9})

NOTEBOOK_DIR = Path('.').resolve()
IMG_SIZE = (200, 200)

DATASET_DIR = None
for _c in [
    NOTEBOOK_DIR / 'dataset' / 'MainData',
    NOTEBOOK_DIR / 'dataset' / 'FullData',
    NOTEBOOK_DIR.parent / 'HLG' / 'dataset' / 'MainData',
    NOTEBOOK_DIR.parent / 'HLG' / 'dataset' / 'FullData',
]:
    if _c.exists() and any(_c.glob('*.jpg')):
        DATASET_DIR = _c
        break

assert DATASET_DIR is not None, 'Khong tim thay dataset!'
print(f'Dataset: {DATASET_DIR}')
print(f'So anh: {len(list(DATASET_DIR.glob("*.jpg")))}')

In [ ]:
# ---- Tai anh mau ----
sample_paths = sorted(DATASET_DIR.glob('*.jpg'))[:3]
samples = []
for p in sample_paths:
    img = cv2.imread(str(p))
    if img is not None:
        samples.append((p.name, img))

def parse_label(name):
    parts = name.split('_')
    if len(parts) < 3:
        return name[:25]
    gender = 'Nam' if parts[1] == '0' else 'Nu'
    race_map = {'0': 'Trang', '1': 'Den', '2': 'A Dong', '3': 'An Do', '4': 'Khac'}
    return f'Tuoi={parts[0]}, {gender}, {race_map.get(parts[2], "?")}'  

fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 4.5))
if len(samples) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, samples):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Anh goc\n{parse_label(name)}', fontsize=10)
    ax.axis('off')
plt.suptitle('Anh mau tu dataset  (TRUOC moi xu ly)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Kich thuoc goc: {samples[0][1].shape[1]}x{samples[0][1].shape[0]} px')

In [ ]:
# ---- Tien xu ly: Resize ve 200x200 ----
name0, img0_bgr = samples[0]
img0_resized = cv2.resize(img0_bgr, IMG_SIZE, interpolation=cv2.INTER_AREA)
img0_rgb     = cv2.cvtColor(img0_resized, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cv2.cvtColor(img0_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'TRUOC  ({img0_bgr.shape[1]}x{img0_bgr.shape[0]} px)', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(img0_rgb)
axes[1].set_title(f'SAU — Resize\n{IMG_SIZE[0]}x{IMG_SIZE[1]} px', fontweight='bold', color='darkred')
axes[1].axis('off')
plt.suptitle('Tien xu ly chung: Resize ve 200x200', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Đặc trưng 1 — HOG (Histogram of Oriented Gradients)

- Chuyển ảnh sang **grayscale**, chia thành các ô (cell) 8×8 pixel
- Tính **histogram gradient** theo 9 hướng trong mỗi ô
- Gom nhóm 2×2 ô thành block, chuẩn hóa L2-Hys
- Kết quả: vector mô tả hình dạng, cạnh và kết cấu khuôn mặt

In [ ]:
img0_gray = cv2.cvtColor(img0_resized, cv2.COLOR_BGR2GRAY)

hog_vec, hog_vis = hog(
    img0_gray,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    feature_vector=True,
    visualize=True,
)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].imshow(img0_rgb)
axes[0].set_title('TRUOC\nAnh mau (200x200)', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img0_gray, cmap='gray')
axes[1].set_title('Trung gian\nAnh xam (grayscale)', fontsize=10)
axes[1].axis('off')

im_hog = axes[2].imshow(hog_vis, cmap='magma')
axes[2].set_title(f'SAU — HOG Visualization\nvector {len(hog_vec)} chieu', fontweight='bold', color='darkred')
axes[2].axis('off')
fig.colorbar(im_hog, ax=axes[2], fraction=0.046, pad=0.04, label='Gradient magnitude')

plt.suptitle('HHL — Dac trung 1: HOG', fontsize=14, fontweight='bold', color='#1a237e')
plt.tight_layout()
plt.show()

print(f'HOG vector: {len(hog_vec)} chieu')
print(f'  orientations=9 | pixels_per_cell=(8,8) | cells_per_block=(2,2) | norm=L2-Hys')
print(f'  Sang den -> phan tich gradient theo {9} huong tren luoi {IMG_SIZE[0]//8}x{IMG_SIZE[1]//8} o')

---
## Đặc trưng 2 — Color Histogram HSV

- Chuyển ảnh sang không gian màu **HSV**
- Tính histogram 3 chiều với **8 bins mỗi kênh** (H×S×V = 8×8×8 = 512 bins)
- Normalize MinMax → giá trị trong **[0, 1]**

In [ ]:
img0_hsv = cv2.cvtColor(img0_resized, cv2.COLOR_BGR2HSV)
h_ch, s_ch, v_ch = cv2.split(img0_hsv)

# --- Buoc 2: Visualize tung kenh H, S, V ---
# H: hien thi bang hue that (S=220, V=210 de nhin ro)
h_vis = cv2.cvtColor(
    cv2.merge([h_ch, np.full_like(h_ch, 220), np.full_like(h_ch, 210)]),
    cv2.COLOR_HSV2RGB
)
s_vis = np.stack([s_ch, s_ch, s_ch], axis=2)
v_vis = np.stack([v_ch, v_ch, v_ch], axis=2)
th = img0_resized.shape[0] // 3
tw = img0_resized.shape[1]
hsv_panel = np.vstack([
    cv2.resize(h_vis,               (tw, th)),
    cv2.resize(s_vis.astype('uint8'), (tw, th)),
    cv2.resize(v_vis.astype('uint8'), (tw, th)),
])

# --- Buoc 3 & 4: Tinh histogram 8 bins / kenh ---
h8 = cv2.calcHist([img0_hsv], [0], None, [8], [0, 180]).flatten()
s8 = cv2.calcHist([img0_hsv], [1], None, [8], [0, 256]).flatten()
v8 = cv2.calcHist([img0_hsv], [2], None, [8], [0, 256]).flatten()

# max_val = gia tri lon nhat trong toan bo 3D histogram (dung cho MinMax)
hist_3d_raw = cv2.calcHist([img0_hsv], [0, 1, 2], None, [8, 8, 8],
                            [0, 180, 0, 256, 0, 256])
max_val = float(hist_3d_raw.max())
h8_norm = h8 / max_val if max_val > 0 else np.zeros(8)
s8_norm = s8 / max_val if max_val > 0 else np.zeros(8)
v8_norm = v8 / max_val if max_val > 0 else np.zeros(8)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# ---- Buoc 1: Anh BGR ----
axes[0].imshow(img0_rgb)
axes[0].set_title('Buoc 1 — TRUOC
Anh BGR (200x200)', fontweight='bold', fontsize=11)
axes[0].axis('off')

# ---- Buoc 2: Cac kenh H / S / V ----
axes[1].imshow(hsv_panel)
for i, (lbl, det) in enumerate([
    ('H  (Hue)', 'goc mau sac'),
    ('S  (Saturation)', 'do bao hoa'),
    ('V  (Value)', 'do sang toi'),
]):
    axes[1].text(tw * 0.5, i * th + th * 0.5,
                 f'{lbl}
{det}',
                 ha='center', va='center', color='white',
                 fontsize=8, fontweight='bold',
                 bbox=dict(facecolor='black', alpha=0.55, pad=2))
for sep in [th, 2 * th]:
    axes[1].axhline(sep, color='white', linewidth=2)
axes[1].set_title('Buoc 2 — Chuyen sang HSV
3 kenh: H / S / V', fontweight='bold', fontsize=11)
axes[1].axis('off')

# ---- Buoc 3: Histogram TRUOC chuan hoa ----
x = np.arange(8)
w = 0.28
axes[2].bar(x - w, h8, w, color='#e53935', alpha=0.85, label='H')
axes[2].bar(x,     s8, w, color='#43a047', alpha=0.85, label='S')
axes[2].bar(x + w, v8, w, color='#1e88e5', alpha=0.85, label='V')
axes[2].set_title('Buoc 3 — Histogram TRUOC chuan hoa
(y = so pixel / bin)', fontweight='bold', fontsize=10)
axes[2].set_xlabel('Bin index (0-7)')
axes[2].set_ylabel('So pixel')
axes[2].set_xticks(range(8))
axes[2].legend(fontsize=9)
axes[2].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{int(v):,}')
)

# ---- Buoc 4: Histogram SAU chuan hoa MinMax -> [0,1] ----
axes[3].bar(x - w, h8_norm, w, color='#e53935', alpha=0.85, label='H')
axes[3].bar(x,     s8_norm, w, color='#43a047', alpha=0.85, label='S')
axes[3].bar(x + w, v8_norm, w, color='#1e88e5', alpha=0.85, label='V')
axes[3].axhline(y=1.0, color='gray', linewidth=1.2,
                linestyle='--', alpha=0.6, label='max = 1.0')
axes[3].set_title('Buoc 4 — Histogram SAU chuan hoa
(MinMax → [0, 1])',
                  fontweight='bold', fontsize=10, color='darkred')
axes[3].set_xlabel('Bin index (0-7)')
axes[3].set_ylabel('Gia tri chuan hoa [0, 1]')
axes[3].set_xticks(range(8))
axes[3].set_ylim(0, 1.18)
axes[3].legend(fontsize=9)

plt.suptitle('HHL — Dac trung 2: Color Histogram HSV (4 buoc)',
             fontsize=14, fontweight='bold', color='#1b5e20')
plt.tight_layout()
plt.show()

hist_3d_norm = (hist_3d_raw / max_val).flatten()
print(f'Color Histogram vector: {len(hist_3d_norm)} chieu  (8x8x8 = 512 bins)')
print(f'  Buoc 3  max raw count : {int(max_val):,} pixels')
print(f'  Buoc 4  sau MinMax    : tat ca gia tri trong [0.0, 1.0]')

---
## Đặc trưng 3 — Facial Landmarks (468 điểm)

- Dùng **MediaPipe Face Mesh** để phát hiện 468 landmark khuôn mặt
- Mỗi điểm lưu tọa độ **(x, y)** chuẩn hóa trong [0, 1]
- Vector kết quả có **468 × 2 = 936 chiều**
- Không phát hiện được mặt → trả về **vector zeros(936)**

In [ ]:
face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1,
    refine_landmarks=False, min_detection_confidence=0.5
)
results = face_mesh.process(img0_rgb)
face_mesh.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

axes[0].imshow(img0_rgb)
axes[0].set_title('TRUOC\nAnh mau (200x200)', fontweight='bold')
axes[0].axis('off')

if results.multi_face_landmarks:
    lm = results.multi_face_landmarks[0].landmark
    h_img, w_img = img0_resized.shape[:2]
    xs_all = [p.x * w_img for p in lm]
    ys_all = [p.y * h_img for p in lm]

    axes[1].imshow(img0_rgb, alpha=0.5)
    axes[1].scatter(xs_all, ys_all, s=2.5, c='lime', alpha=0.85, linewidths=0, label='468 pts')

    key_groups = [
        ([33, 133, 159, 145],  'cyan',        'Mat trai'),
        ([263, 362, 386, 374], 'dodgerblue',  'Mat phai'),
        ([6, 1],               'yellow',      'Mui'),
        ([61, 291],            'red',         'Mieng'),
        ([234, 454, 10, 152],  'white',       'Vien mat'),
    ]
    for idxs, color, label in key_groups:
        kx = [lm[i].x * w_img for i in idxs]
        ky = [lm[i].y * h_img for i in idxs]
        axes[1].scatter(kx, ky, s=35, c=color, zorder=7, label=label,
                        edgecolors='black', linewidths=0.5)

    axes[1].legend(loc='lower right', fontsize=8, facecolor='black',
                   labelcolor='white', framealpha=0.75)
    axes[1].set_title(f'SAU — 468 Landmarks\nvector {468*2} chieu (x,y moi diem)',
                      fontweight='bold', color='darkred')
else:
    axes[1].imshow(img0_rgb)
    axes[1].set_title('Khong phat hien khuon mat\n-> zeros(936)', color='red')

axes[1].axis('off')
plt.suptitle('HHL — Dac trung 3: Facial Landmarks (MediaPipe Face Mesh)',
             fontsize=14, fontweight='bold', color='#1a237e')
plt.tight_layout()
plt.show()

status = 'Phat hien duoc' if results.multi_face_landmarks else 'Khong phat hien'
print(f'Ket qua: {status}')
print(f'Landmark vector: 468 x 2 = {468*2} chieu')
print(f'  Moi diem: (x, y) chuan hoa [0,1] theo kich thuoc anh')

---
## Tổng hợp: 3 ảnh mẫu × 3 đặc trưng HHL

In [ ]:
col_titles = ['Anh goc\n(TRUOC)', 'HOG\n(~3780d)', 'Color Hist HSV\n(512d)', 'Landmarks\n(936d)']

fig, axes = plt.subplots(len(samples), 4, figsize=(18, 4.5 * len(samples)))
if len(samples) == 1:
    axes = [axes]

face_mesh2 = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1,
    refine_landmarks=False, min_detection_confidence=0.5
)

for row, (name_s, img_s) in enumerate(samples):
    img_r = cv2.resize(img_s, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img_rgb_s = cv2.cvtColor(img_r, cv2.COLOR_BGR2RGB)
    img_gray_s = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)
    img_hsv_s  = cv2.cvtColor(img_r, cv2.COLOR_BGR2HSV)

    # Col 0: original
    axes[row][0].imshow(img_rgb_s)
    axes[row][0].set_ylabel(parse_label(name_s), fontsize=8, rotation=0,
                             labelpad=65, verticalalignment='center')

    # Col 1: HOG
    _, hog_v = hog(img_gray_s, orientations=9, pixels_per_cell=(8,8),
                   cells_per_block=(2,2), block_norm='L2-Hys',
                   feature_vector=True, visualize=True)
    axes[row][1].imshow(hog_v, cmap='magma')

    # Col 2: Color Histogram (8 bins per channel, bar chart)
    h8s = cv2.calcHist([img_hsv_s],[0],None,[8],[0,180]).flatten()
    s8s = cv2.calcHist([img_hsv_s],[1],None,[8],[0,256]).flatten()
    v8s = cv2.calcHist([img_hsv_s],[2],None,[8],[0,256]).flatten()
    xb = np.arange(8)
    axes[row][2].bar(xb-0.27, h8s/max(h8s.max(),1), 0.27, color='#e53935', alpha=0.85, label='H')
    axes[row][2].bar(xb,      s8s/max(s8s.max(),1), 0.27, color='#43a047', alpha=0.85, label='S')
    axes[row][2].bar(xb+0.27, v8s/max(v8s.max(),1), 0.27, color='#1e88e5', alpha=0.85, label='V')
    axes[row][2].set_xticks(range(8))
    if row == 0:
        axes[row][2].legend(fontsize=7)

    # Col 3: Landmarks
    res_s = face_mesh2.process(img_rgb_s)
    axes[row][3].imshow(img_rgb_s, alpha=0.5)
    if res_s.multi_face_landmarks:
        lm_s = res_s.multi_face_landmarks[0].landmark
        h_s, w_s = img_r.shape[:2]
        axes[row][3].scatter(
            [p.x * w_s for p in lm_s],
            [p.y * h_s for p in lm_s],
            s=1.5, c='lime', alpha=0.8, linewidths=0
        )

    for col in range(4):
        if col in (0, 1, 3):
            axes[row][col].axis('off')

face_mesh2.close()

for col, title in enumerate(col_titles):
    axes[0][col].set_title(title, fontsize=10, fontweight='bold')

plt.suptitle('HHL — Tong hop TRUOC va SAU cho ca 3 dac trung',
             fontsize=14, fontweight='bold', color='#1a237e', y=1.01)
plt.tight_layout()
plt.show()

print('\nTong chieu vector HHL:')
print(f'  HOG:              ~3780 chieu')
print(f'  Color Hist HSV:    512 chieu')
print(f'  Landmarks:         936 chieu')
print(f'  TONG:             ~5228 chieu')